In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from matplotlib_venn import venn2

ModuleNotFoundError: No module named 'matplotlib_venn'

In [2]:
# Change for your data
root_path = Path("/data/smfish/20260311_bartelle_smFISH_cryo_48hr_male")

In [3]:
results_dir = (
        root_path
        / "big_fish"
        / "results"
        / "all_tiles_2D"
    )

In [4]:
# -------------------------------------------------------------------------
# Load spots from all 16 bits
# -------------------------------------------------------------------------
all_spots = []

for bit in range(1, 17):

    spots_path = results_dir / f"spots_bit_{bit}.csv"

    if not spots_path.exists():
        print(f"WARNING: {spots_path.name} not found. Skipping.")
        continue

    spots = pd.read_csv(spots_path, index_col=0)
    print(spots.head())

    if "cell_id" not in spots.columns:
        raise ValueError(
            f"{spots_path.name} does not contain a cell_id column."
        )

    # Explicitly record bit number.
    # This avoids depending on how the bit_# column is formatted.
    spots["bit"] = bit

    all_spots.append(spots)

    print(f"Bit {bit:2d}: {len(spots):,} spots")

all_spots = pd.concat(all_spots, ignore_index=True)

print()
# print(f"Total detected spots: {len(all_spots):,}")

           y     x  bit  cell_id
spot_id                         
0        311  3657    1        0
1        338  4340    1       10
2        473  4679    1        0
3        596  7843    1        0
4        652  4414    1       69
Bit  1: 4,389 spots
           y     x  bit  cell_id
spot_id                         
0         52  1967    2        1
1        116  3358    2        6
2        120  3373    2        6
3        120  3405    2        6
4        121  2971    2        0
Bit  2: 408,603 spots
           y     x  bit  cell_id
spot_id                         
0         85  2669    3        0
1         97  2763    3        0
2        132  3377    3        6
3        201  4382    3       10
4        232  4811    3       15
Bit  3: 31,452 spots
           y     x  bit  cell_id
spot_id                         
0        126  2736    4        0
1        162  3372    4        6
2        164  3348    4        6
3        168  3437    4        6
4        172  3347    4        6
Bit  4: 258,9

In [8]:
# Create a set of unique (x, y) coordinates for each bit
coordinate_sets = {
    bit_value: set(
        zip(
            group["x"].astype(int),
            group["y"].astype(int),
        )
    )
    for bit_value, group in all_spots.groupby("bit")
}

bits = sorted(coordinate_sets)

# Check repeated coordinates within each bit
within_bit_summary = pd.DataFrame(
    [
        {
            "bit": bit_value,
            "spots": len(all_spots[all_spots["bit"] == bit_value]),
            "unique_coordinates": len(coordinate_sets[bit_value]),
            "duplicate_spots": (
                len(all_spots[all_spots["bit"] == bit_value])
                - len(coordinate_sets[bit_value])
            ),
        }
        for bit_value in bits
    ]
)

display(within_bit_summary)

# Calculate pairwise coordinate intersections
pairwise_overlap = []

for bit_a, bit_b in combinations(bits, 2):
    set_a = coordinate_sets[bit_a]
    set_b = coordinate_sets[bit_b]
    intersection = set_a & set_b
    union = set_a | set_b

    pairwise_overlap.append(
        {
            "bit_a": bit_a,
            "bit_b": bit_b,
            "coordinates_bit_a": len(set_a),
            "coordinates_bit_b": len(set_b),
            "overlap": len(intersection),
            "overlap_percent_of_bit_a": 100 * len(intersection) / len(set_a),
            "overlap_percent_of_bit_b": 100 * len(intersection) / len(set_b),
            "jaccard_percent": 100 * len(intersection) / len(union),
        }
    )

pairwise_overlap = pd.DataFrame(pairwise_overlap)

# Display pairs with the greatest overlap
display(
    pairwise_overlap.sort_values(
        "overlap",
        ascending=False,
    ).head(20)
)

# Create an overlap-count matrix
overlap_matrix = pd.DataFrame(
    0,
    index=bits,
    columns=bits,
    dtype=int,
)

for bit_a, bit_b in combinations(bits, 2):
    overlap = len(coordinate_sets[bit_a] & coordinate_sets[bit_b])
    overlap_matrix.loc[bit_a, bit_b] = overlap
    overlap_matrix.loc[bit_b, bit_a] = overlap

np.fill_diagonal(
    overlap_matrix.values,
    [len(coordinate_sets[bit_value]) for bit_value in bits],
)


,bit,spots,unique_coordinates,duplicate_spots
0,1,4389,4389,0
1,2,408603,408603,0
2,3,31452,31452,0
3,4,258931,258931,0
4,5,20905,20905,0
5,6,15655,15655,0
6,7,43689,43689,0
7,8,12322,12322,0
8,9,10615,10615,0
9,10,162332,162332,0


,bit_a,bit_b,coordinates_bit_a,coordinates_bit_b,overlap,overlap_percent_of_bit_a,overlap_percent_of_bit_b,jaccard_percent
16,2,4,408603,258931,1997,0.488738,0.771248,0.300058
43,4,6,258931,15655,1240,0.478892,7.920792,0.453638
85,8,10,12322,162332,1104,8.959584,0.680088,0.636128
18,2,6,408603,15655,827,0.202397,5.282657,0.195309
118,14,16,22699,26237,720,3.171946,2.744216,1.493280
111,12,14,11907,22699,586,4.921475,2.581612,1.722516
113,12,16,11907,26237,513,4.308390,1.955254,1.363238
30,3,5,31452,20905,320,1.017423,1.530734,0.614947
76,7,9,43689,10615,294,0.672938,2.769666,0.544344
1,1,3,4389,31452,269,6.128959,0.855272,0.756213


ValueError: underlying array is read-only

In [ ]:
# Plot pairwise overlap heatmap
fig, ax = plt.subplots(figsize=(8, 7))

image = ax.imshow(overlap_matrix, cmap="magma")

ax.set_xticks(range(len(bits)))
ax.set_yticks(range(len(bits)))
ax.set_xticklabels(bits)
ax.set_yticklabels(bits)
ax.set_xlabel("Bit")
ax.set_ylabel("Bit")
ax.set_title("Exact coordinate overlap between bits")

for row in range(len(bits)):
    for col in range(len(bits)):
        ax.text(
            col,
            row,
            overlap_matrix.iloc[row, col],
            ha="center",
            va="center",
            color="white",
            fontsize=8,
        )

fig.colorbar(image, ax=ax, label="Number of shared coordinates")
plt.tight_layout()
plt.show()

# Venn diagram for the pair with the greatest overlap
top_pair = pairwise_overlap.loc[
    pairwise_overlap["overlap"].idxmax()
]

bit_a = int(top_pair["bit_a"])
bit_b = int(top_pair["bit_b"])

try:

    plt.figure(figsize=(5, 5))
    venn2(
        [
            coordinate_sets[bit_a],
            coordinate_sets[bit_b],
        ],
        set_labels=(f"Bit {bit_a}", f"Bit {bit_b}"),
    )
    plt.title(f"Coordinate overlap: Bit {bit_a} vs Bit {bit_b}")
    plt.show()

except ImportError:
    print(
        "Install matplotlib-venn to create Venn diagrams: "
        "pip install matplotlib-venn"
    )